In [1]:
!pip install -q polars faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.8 MB/s eta 0:00:00


In [2]:
import os
import gc
import shutil
import pandas as pd
import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
import faiss
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang sử dụng thiết bị: {device}")

Đang sử dụng thiết bị: cuda


In [3]:
DATASET_DIR_NAME = 'datasets/b22dckh072/file02' 

INPUT_DIR = f'/kaggle/input/{DATASET_DIR_NAME}'
WORKING_DIR = '/kaggle/working'

TRAIN_PATH = os.path.join(INPUT_DIR, 'train_interactions.parquet')

LIGHTGCN_CAND_PATH = os.path.join(WORKING_DIR, 'lightgcn_candidates.parquet')
MAX_LEN   = 50

def load_data(path):
    df = pl.read_parquet(
        path,
        columns=['mapped_user_id', 'mapped_item_id', 'rating', 'timestamp']
    ).to_pandas()
    return df

In [4]:
torch.cuda.empty_cache()
gc.collect()

30

In [5]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import scipy.sparse as sp
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import pandas as pd
import numpy as np

torch.cuda.empty_cache()
gc.collect()

print("Đang chuẩn bị dữ liệu LightGCN...")
df_train_pd = load_data(TRAIN_PATH)
u_idx, i_idx = df_train_pd['mapped_user_id'].to_numpy(), df_train_pd['mapped_item_id'].to_numpy()

num_users = df_train_pd['mapped_user_id'].max() + 1
num_items = df_train_pd['mapped_item_id'].max() + 1

# Tạo ma trận kề COO bằng SciPy
adj = sp.coo_matrix((np.ones(len(u_idx)), (u_idx, i_idx + num_users)), shape=(num_users+num_items, num_users+num_items))
adj = adj + adj.T

# Chuẩn hóa ma trận
d_inv = np.power(np.array(adj.sum(1)), -0.5).flatten()
d_inv[np.isinf(d_inv)] = 0.
d_mat = sp.diags(d_inv)

print("Đang nén đồ thị sang định dạng PyTorch CSR chuẩn (int64)...")
norm_adj_csr = d_mat.dot(adj).dot(d_mat).tocsr()

crow_indices = torch.tensor(norm_adj_csr.indptr, dtype=torch.long)
col_indices = torch.tensor(norm_adj_csr.indices, dtype=torch.long)
values = torch.tensor(norm_adj_csr.data, dtype=torch.float32)

norm_adj_t = torch.sparse_csr_tensor(crow_indices, col_indices, values, size=norm_adj_csr.shape).to(device)

del adj, d_inv, d_mat, norm_adj_csr
gc.collect()


EMBED_DIM_LGCN = 64

class LightGCN(nn.Module):
    def __init__(self, u, i, dim):
        super().__init__()
        self.u_emb = nn.Embedding(u, dim)
        self.i_emb = nn.Embedding(i, dim)
        nn.init.normal_(self.u_emb.weight, std=0.1)
        nn.init.normal_(self.i_emb.weight, std=0.1)
        
    def forward(self, adj):
        emb0 = torch.cat([self.u_emb.weight, self.i_emb.weight])
        e1 = torch.sparse.mm(adj, emb0)
        e2 = torch.sparse.mm(adj, e1)
        
        return torch.split((emb0 + e1 + e2) / 3.0, [num_users, num_items])

model_lgcn = LightGCN(num_users, num_items, EMBED_DIM_LGCN).to(device)

# Đã loại bỏ weight_decay để tránh phá hủy lớp Embedding
optimizer = torch.optim.Adam(model_lgcn.parameters(), lr=0.001)

pos_pairs = df_train_pd[['mapped_user_id', 'mapped_item_id']].to_numpy()
batch_size_lgcn = 204800

print("Đang chuẩn bị Tensor trên RAM cho LightGCN...")
pos_pairs_tensor = torch.tensor(pos_pairs, dtype=torch.long)

MAX_EPOCHS = 200      
PATIENCE = 10         
MIN_DELTA = 0.001     

best_train_loss = float('inf')
epochs_no_improve = 0

for ep in range(MAX_EPOCHS):
    idx_perm = torch.randperm(len(pos_pairs_tensor))
    loss_ep, t_batches = 0, 0
    pbar = tqdm(range(0, len(pos_pairs_tensor), batch_size_lgcn), desc=f"LightGCN Epoch {ep+1}/{MAX_EPOCHS}", leave=False)

    for i in pbar:
        b_idx = idx_perm[i:i+batch_size_lgcn]
        batch = pos_pairs_tensor[b_idx].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        u_reps, i_reps = model_lgcn(norm_adj_t)

        users = batch[:, 0]
        pos_items = batch[:, 1]
        neg_items = torch.randint(0, num_items, (len(batch),), device=device)

        u_emb = u_reps[users]
        pos_emb = i_reps[pos_items]
        neg_emb = i_reps[neg_items]

        u_emb_0 = model_lgcn.u_emb(users)
        pos_emb_0 = model_lgcn.i_emb(pos_items)
        neg_emb_0 = model_lgcn.i_emb(neg_items)

        del u_reps, i_reps

        pos_scores = (u_emb * pos_emb).sum(1)
        neg_scores = (u_emb * neg_emb).sum(1)

        bpr_loss = -F.logsigmoid(pos_scores - neg_scores).mean()

        reg_loss = (1/2) * (u_emb_0.norm(2).pow(2) + pos_emb_0.norm(2).pow(2) + neg_emb_0.norm(2).pow(2)) / float(len(users))
        loss = bpr_loss + 1e-4 * reg_loss

        del u_emb, pos_emb, neg_emb, pos_scores, neg_scores, u_emb_0, pos_emb_0, neg_emb_0

        loss.backward()
        optimizer.step()

        loss_ep += loss.item()
        t_batches += 1
        pbar.set_postfix(loss=loss_ep/t_batches)

    avg_loss = loss_ep / t_batches
    print(f"LightGCN Epoch {ep+1} | Train Loss: {avg_loss:.4f}")
    
    if (best_train_loss - avg_loss) > MIN_DELTA:
        best_train_loss = avg_loss
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"  -> Loss không giảm đáng kể (Patience: {epochs_no_improve}/{PATIENCE})")
        if epochs_no_improve >= PATIENCE:
            print(f"KÍCH HOẠT EARLY STOPPING TẠI EPOCH {ep+1}!")
            break

Đang chuẩn bị dữ liệu LightGCN...


/tmp/ipykernel_23/2516856979.py:28: RuntimeWarning: divide by zero encountered in power
  d_inv = np.power(np.array(adj.sum(1)), -0.5).flatten()


Đang nén đồ thị sang định dạng PyTorch CSR chuẩn (int64)...


/tmp/ipykernel_23/2516856979.py:39: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  norm_adj_t = torch.sparse_csr_tensor(crow_indices, col_indices, values, size=norm_adj_csr.shape).to(device)


Đang chuẩn bị Tensor trên RAM cho LightGCN...


...

LightGCN Epoch 183 | Train Loss: 0.0179
  -> Loss không giảm đáng kể (Patience: 8/10)


LightGCN Epoch 184/200:   0%|          | 0/81 [00:00<?, ?it/s]

LightGCN Epoch 184 | Train Loss: 0.0178
  -> Loss không giảm đáng kể (Patience: 9/10)


LightGCN Epoch 185/200:   0%|          | 0/81 [00:00<?, ?it/s]

LightGCN Epoch 185 | Train Loss: 0.0176
  -> Loss không giảm đáng kể (Patience: 10/10)
KÍCH HOẠT EARLY STOPPING TẠI EPOCH 185!


In [6]:
import os
import gc
import polars as pl
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm
torch.cuda.empty_cache()
gc.collect()

model_lgcn.eval()
infer_batch_size = 512
chunk_size = 1000 

print(f"Đang truy xuất Top 200 LightGCN trực tiếp trên GPU (Batch size: {infer_batch_size})...")

# Tạo thư mục tạm để chứa các phần nhỏ
os.makedirs('/kaggle/working/lightgcn_chunks', exist_ok=True)

all_top_idx_lgcn = []
chunk_user_ids = []
chunk_idx = 0

with torch.no_grad():
    u_e, i_e = model_lgcn(norm_adj_t)

    # 3. Lặp qua từng batch của User
    pbar = tqdm(range(0, num_users, infer_batch_size), desc="Inference LightGCN Native PyTorch")
    for i in pbar:
        # Lấy một nhóm user nhỏ
        u_batch = u_e[i:i+infer_batch_size]
        scores = torch.matmul(u_batch, i_e.T)

        _, top_idx = torch.topk(scores, 200, dim=1)

        # Đẩy kết quả về RAM
        all_top_idx_lgcn.append(top_idx.cpu().numpy().astype('int32'))
        
        # Tạo mảng ID user cho batch hiện tại
        batch_u_ids = np.arange(i, min(i + infer_batch_size, num_users))
        chunk_user_ids.append(batch_u_ids)

        # Ép xóa các Tensor lớn ngay lập tức
        del scores, u_batch, top_idx
        if len(all_top_idx_lgcn) >= chunk_size or (i + infer_batch_size) >= num_users:
            u_ids_arr = np.concatenate(chunk_user_ids)
            item_ids_arr = np.vstack(all_top_idx_lgcn).flatten()
            
            df_chunk = pd.DataFrame({
                'mapped_user_id': np.repeat(u_ids_arr, 200).astype('int32'),
                'mapped_item_id': item_ids_arr.astype('int32'),
                'lightgcn_rank': np.tile(np.arange(1, 201, dtype=np.int16), len(u_ids_arr))
            })
            
            chunk_path = f'/kaggle/working/lightgcn_chunks/chunk_{chunk_idx}.parquet'
            df_chunk.to_parquet(chunk_path)
            
            # Dọn dẹp RAM cho chunk tiếp theo
            del df_chunk, u_ids_arr, item_ids_arr
            all_top_idx_lgcn = []
            chunk_user_ids = []
            chunk_idx += 1
            gc.collect()

# Dọn dẹp VRAM
print("Đang dọn dẹp VRAM GPU...")
del u_e, i_e
torch.cuda.empty_cache()
gc.collect()

# Sử dụng Polars để gộp các file parquet nhỏ lại thành 1 file duy nhất mà không tràn RAM
print("Đang gộp các file nhỏ lại (không tốn RAM)...")
LIGHTGCN_CAND_PATH = '/kaggle/working/lightgcn_candidates.parquet'

lf_lightgcn = pl.scan_parquet('/kaggle/working/lightgcn_chunks/chunk_*.parquet')
lf_lightgcn.sink_parquet(LIGHTGCN_CAND_PATH)

print(f'Đã lưu kết quả LightGCN hoàn chỉnh vào: {LIGHTGCN_CAND_PATH}')

Đang truy xuất Top 200 LightGCN trực tiếp trên GPU (Batch size: 512)...


Inference LightGCN Native PyTorch:   0%|          | 0/4409 [00:00<?, ?it/s]

Đang dọn dẹp VRAM GPU...
Đang gộp các file nhỏ lại (không tốn RAM)...
Đã lưu kết quả LightGCN hoàn chỉnh vào: /kaggle/working/lightgcn_candidates.parquet
